# 🔧 Lab 3 — Build & Deploy the Maintenance Cockpit

Welcome to the fun part. 🎉

In **Lab 1** you created the factory's data, and in **Lab 2** you served it from **Lakebase** — a real Postgres database sitting right next to the lakehouse. Now you'll put a **real app** in front of it: a *Maintenance Cockpit* that a technician on the shop floor uses to see what needs attention and log what they fixed.

You'll do the whole thing **from your browser** — no laptop setup, no command line. Just run the cells from top to bottom and read the short notes as you go.

⏱️ **Time:** ~30 minutes
🎓 **You'll leave knowing how to:** deploy a Databricks App, connect it to Lakebase, and ship a change — the everyday app lifecycle.

## 🗺️ What you're building

A small web app with four tabs — **🔴 Alerts & actions**, **📋 Work orders**, **✅ Quality checks**, **📝 Operator notes** — all reading and writing live to *your* Lakebase database.

```
   Your Lakebase (Postgres)              The Maintenance Cockpit (app)
   ┌────────────────────────┐           ┌───────────────────────────┐
   │ machines, tickets  …   │ ──read──▶ │  🔴 shows what needs fixing│
   │   (synced in Lab 2)    │           │                            │
   │ maintenance_actions …  │ ◀─write── │  ✍️  log the fix you did   │
   └────────────────────────┘           └───────────────────────────┘
```

Almost all the code is written for you. **The one piece you'll finish** is the little function that *saves* a technician's fix back to the database — that's the payoff in Step 6. 💪

> 💡 **Not a hardcore coder? Perfect.** Every step starts with a plain-English "what this does". You mostly read, run, and watch it come to life.

## Step 1 — Warm up 🐍

First we install the two helpers the notebook needs and restart Python so they load cleanly.

> ⚠️ `restartPython()` clears everything, so **run the next cell right after it** — that one sets up all our variables again.

In [ ]:
%pip install -U "databricks-sdk>=0.118.0" "psycopg[binary]>=3.1" -q

In [ ]:
dbutils.library.restartPython()

## Step 2 — Find your database and point the app at it 🧭

Everyone in the workshop has their **own** Lakebase project from Lab 2 (named `lakebase-ws-<you>-N`). This cell finds it automatically and writes your connection details into the app's config file (`app.yaml`) — so you never edit anything by hand.

👉 **The only thing you set:** `REPO` — the path to your Git folder (right-click the repo in the sidebar ▸ *Copy URL/path*).

In [ ]:
import re, time, pathlib
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import App, AppDeployment
import psycopg

# Find the app automatically — search from this notebook's folder upward for bundle/src/app.
# (Works whether the notebook sits in <repo>/notebooks/ or right next to bundle/. Nothing to type.)
_dir = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().rsplit("/", 1)[0]
REPO = None
for _ in range(5):
    _cand = _dir if _dir.startswith("/Workspace") else "/Workspace" + _dir
    if pathlib.Path(f"{_cand}/bundle/src/app").exists():
        REPO = _cand
        break
    if _dir.count("/") <= 1:
        break
    _dir = _dir.rsplit("/", 1)[0]
assert REPO, ("Couldn't find bundle/src/app near this notebook. Make sure the repo's bundle/ "
              "folder sits next to this notebook (or one level up).")
APP_SRC = f"{REPO}/bundle/src/app"

w = WorkspaceClient()
user = w.current_user.me().user_name
slug = re.sub(r"[^a-z0-9]", "", user.split("@")[0].lower())

# Find the Lakebase project you created in Lab 2 (named lakebase-ws-<you>-N)
PROJECT = None
for i in range(1, 21):
    cand = f"lakebase-ws-{slug}-{i}"
    try:
        w.postgres.get_project(name=f"projects/{cand}")
        list(w.postgres.list_branches(parent=f"projects/{cand}"))   # is it healthy?
        PROJECT = cand
        break
    except Exception:
        continue
assert PROJECT, "No Lakebase project found — did you finish Lab 2?"

BRANCH   = f"projects/{PROJECT}/branches/production"
ENDPOINT = f"{BRANCH}/endpoints/primary"
PGDB     = "databricks_postgres"          # your operational tables live here (public schema)
LBSCHEMA = f"lakebase_{slug}"             # your read-only synced tables live here
APP      = ("lb-workshop-" + slug)[:30].rstrip("-")
host = next(e for e in w.postgres.list_endpoints(parent=BRANCH)
            if e.name == ENDPOINT).status.hosts.host

# Fill your endpoint + host into the app's config file (no hand-editing needed)
def _set(t, k, v):
    return re.sub(rf'(- name: {k}\n\s*value: )"[^"]*"', rf'\g<1>"{v}"', t)
cfg = pathlib.Path(f"{APP_SRC}/app.yaml")
txt = cfg.read_text()
txt = _set(txt, "ENDPOINT_NAME", ENDPOINT)
txt = _set(txt, "PGHOST", host)
cfg.write_text(txt)

print(f"✅ Repo detected:               {REPO}")
print(f"✅ Found your Lakebase project: {PROJECT}")
print(f"   Your app will be named:     {APP}")
print(f"   Synced tables are in schema: {LBSCHEMA}")
print(f"   App config points at:        {host}")

## Step 3 — Create the app and connect it to your database 🔌

This creates the app and **binds** your Lakebase database to it. "Binding" simply gives the app its own login to Postgres — so it can talk to your data securely, without you pasting any passwords.

*(First run provisions a little compute — about 2 minutes. Grab a coffee. ☕)*

In [ ]:
# Which Postgres database object to bind (the one Lab 2 created)
dbres = next(d.as_dict()["name"] for d in w.postgres.list_databases(BRANCH)
             if d.as_dict()["status"]["postgres_database"] == PGDB)

try:
    w.apps.get(name=APP)
    print(f"✅ App '{APP}' already exists — skipping create")
except Exception:
    print(f"⏳ Creating '{APP}' and giving it a little compute — usually ~2 minutes. "
          f"You'll see progress below (the notebook stays responsive).")
    # Kick off creation WITHOUT blocking the kernel, then poll politely.
    w.apps.create(App.from_dict({
        "name": APP,
        "description": "Maintenance Cockpit — Lakebase in a Day",
        "resources": [{
            "name": "lakebase-db",
            "postgres": {"branch": BRANCH, "database": dbres,
                         "permission": "CAN_CONNECT_AND_CREATE"},
        }],
    }))
    for _ in range(90):
        try:
            state = w.apps.get(name=APP).as_dict().get("compute_status", {}).get("state")
        except Exception:
            state = "PROVISIONING"
        if state == "ACTIVE":
            break
        print(f"   compute: {state} …")
        time.sleep(10)
    print("✅ App created and connected to your Lakebase database")

## Step 4 — Give the app permission 🔑

The app connects as its own identity (a *service principal*). We grant it **read** access to your synced tables and **read + write** to the operational ones — once. That's all: the app takes care of finding tables across both schemas (`public` and `lakebase_<you>`) itself.

> 💡 If the app ever shows `insufficientPrivilege`, these grants are the fix.

In [ ]:
# The app runs as its own identity (a "service principal"). Give that identity read access to
# your synced tables and read+write to your operational ones — once.
#   • pg_read_all_data / pg_write_all_data cover every table, now and after a re-sync
#   • the sequence grant lets inserts use the SERIAL primary keys
# (You don't need to touch search_path here — the app sets its own per connection, so it finds
#  tables in both the public and lakebase_<you> schemas automatically.)
sp = w.apps.get(name=APP).as_dict()["service_principal_client_id"]

with psycopg.connect(host=host, port=5432, dbname=PGDB, user=user,
                     password=w.postgres.generate_database_credential(ENDPOINT).token,
                     sslmode="require", autocommit=True) as c:
    c.execute(f'GRANT USAGE ON SCHEMA public TO "{sp}"')
    c.execute(f'GRANT pg_read_all_data  TO "{sp}"')
    c.execute(f'GRANT pg_write_all_data TO "{sp}"')
    c.execute(f'GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO "{sp}"')

print("✅ Permissions granted — the app can read your synced tables and read/write the operational ones.")

## Step 5 — Deploy! 🚀

Ship the app. When it finishes you'll get a URL.

**✅ Check:** the deploy reaches **SUCCEEDED** and prints a link. Open it — you'll see the four tabs and a list of **open alerts** pulled live from Lakebase (e.g. a high-priority ticket on a `CNC-X1000`). Try the **"Log action"** button — it politely says *"not implemented yet."* That's on purpose: **you'll fix it in the next step.** 😉

In [ ]:
# Deploy WITHOUT blocking the kernel (no *_and_wait), then poll politely so the notebook
# stays responsive and shows progress.
_prev = (w.apps.get(name=APP).as_dict().get("active_deployment") or {}).get("deployment_id")
w.apps.deploy(app_name=APP,
              app_deployment=AppDeployment.from_dict({"source_code_path": APP_SRC}))
print("⏳ Deploying your app — usually under a minute…")
_url = None
for _ in range(90):
    d = w.apps.get(name=APP).as_dict()
    ad = d.get("active_deployment") or {}
    state = (ad.get("status") or {}).get("state")
    if ad.get("deployment_id") != _prev and state == "SUCCEEDED":
        _url = d.get("url"); break
    if state == "FAILED":
        raise RuntimeError("Deploy failed: " + str((ad.get("status") or {}).get("message")))
    print("   …deploying")
    time.sleep(8)
print("✅ Deployed!")
print("🌐 Open your app:", _url or w.apps.get(name=APP).url)

## Step 6 — Finish the write-back (the payoff) ✍️

Right now the app can *show* alerts but can't *save* a technician's work — because one function, `log_maintenance_action()` in `db.py`, is left blank on purpose.

Below is the finished version. Read it (it's just one `INSERT`), then run the cell to drop it in. Want a challenge? Try writing it yourself first, then compare.

> 🧠 **What it does, in words:** add one row to `maintenance_actions` — which machine, which ticket, what was done, by whom, and (if the job is complete) the timestamp.

In [ ]:
# This is the ONE bit of logic that's yours to finish. Open bundle/src/app/db.py and look at
# log_maintenance_action() — right now it just raises NotImplementedError. Below is the finished
# version. (Try writing it yourself first if you like, then run this cell to drop it in.)
#
# In plain English: it inserts one row into the maintenance_actions table — which machine, which
# ticket, what was done, by whom, and (if the job is finished) the completion time.

impl = '''    with conn.cursor() as cur:
        cur.execute(
            """INSERT INTO maintenance_actions
                   (machine_id, ticket_id, action_type, description, performed_by, status, completed_at)
               VALUES (%s, %s, %s, %s, %s, %s, CASE WHEN %s = 'completed' THEN now() END)""",
            (machine_id, ticket_id, action_type, description, performed_by, status, status))
    conn.commit()'''

stub = '    raise NotImplementedError("Not implemented yet — see Lab 3, Step 4.")'
dbpy = pathlib.Path(f"{APP_SRC}/db.py")
src = dbpy.read_text()
if stub in src:
    dbpy.write_text(src.replace(stub, impl))
    print("✅ Write-back implemented — log_maintenance_action() now saves the technician's work.")
elif "INSERT INTO maintenance_actions" in src:
    print("✅ Write-back is already implemented — nothing to change. (Re-running is fine.)")
else:
    print("⚠️ Couldn't find the stub or an existing implementation in db.py — take a look before deploying.")

## Step 7 — Redeploy and see it work 🎬

Push your change and reload the app.

**✅ Check:** in the app, pick an alert, write what you did (e.g. *"replaced coolant filter"*), and hit **Log action**. It now saves! Your entry shows up under **"Recent maintenance actions."** You just closed the loop between the shop floor and the database. 🙌

In [ ]:
# Same non-blocking deploy as before — push your write-back and wait for it to go live.
_prev = (w.apps.get(name=APP).as_dict().get("active_deployment") or {}).get("deployment_id")
w.apps.deploy(app_name=APP,
              app_deployment=AppDeployment.from_dict({"source_code_path": APP_SRC}))
print("⏳ Redeploying with your change — usually under a minute…")
_url = None
for _ in range(90):
    d = w.apps.get(name=APP).as_dict()
    ad = d.get("active_deployment") or {}
    state = (ad.get("status") or {}).get("state")
    if ad.get("deployment_id") != _prev and state == "SUCCEEDED":
        _url = d.get("url"); break
    if state == "FAILED":
        raise RuntimeError("Deploy failed: " + str((ad.get("status") or {}).get("message")))
    print("   …deploying")
    time.sleep(8)
print("✅ Redeployed! Reload the app and try 'Log action' again.")
print("🌐", _url or w.apps.get(name=APP).url)

## 🎉 You did it!

You just took an app through its whole life — **all in the browser**:

1. 🧭 pointed it at your Lakebase database
2. 🔌 created & connected the app
3. 🔑 granted it access (and fixed the `search_path` gotcha)
4. 🚀 deployed it
5. ✍️ finished the write-back and shipped the change

Your app now serves live operational data *and* records what happens on the floor.

### What's next
➡️ **Lab 4 — Close the round-trip.** With **Change Data Feed** turned on, every action your app writes flows straight back into the lakehouse as a Delta table — queryable in Databricks SQL, ready for reporting and the breakdown-prediction model. That's the full circle: from an alert, to a fix on the floor, back to analytics.